In [1]:
import pandas as pd
from pathlib import Path

#数据读取
# 指定数据类型，大幅压缩内存
dtypes = {
    'user_id':       'int32',
    'item_id':       'int32',
    'category_id':   'int32',
    'behavior_type': 'category',
    'timestamp':     'int32',
}

data_path = Path.cwd().parent / 'UserBehavior.csv'

df = pd.read_csv(
    data_path,
    names=['user_id', 'item_id', 'category_id', 'behavior_type', 'timestamp'],
    dtype=dtypes
)

print(f"总数据量: {len(df)}")

总数据量: 100150807


In [2]:
#缺失值,重复值,异常值处理
df = df.dropna(how='any',axis=0)

df = df.drop_duplicates(subset=['user_id', 'item_id','category_id','behavior_type','timestamp'])

behavior = {'pv','cart','fav','buy'}
df = df[df['behavior_type'].isin(behavior)]

df['datetime'] = pd.to_datetime(df['timestamp'],unit='s')
start,end = pd.to_datetime('2017-11-25'),pd.to_datetime('2017-12-3')
df = df[df['datetime'].between(start,end)]

user_pv_count = df[df['behavior_type'] == 'pv'].groupby('user_id').size()
users = user_pv_count[(user_pv_count > 0) & (user_pv_count < 1000)].index
df = df[df['user_id'].isin(users)]

print(f'清洗后的数据量：{len(df)}条')

清洗后的数据量：86870392条


In [3]:
#EDA
# 以用户为粒度统计每种行为的 UV（去重用户数）
funnel_uv = (
    df.groupby("behavior_type", observed= False)["user_id"]
    .nunique()
    .reindex(["pv", "fav", "cart", "buy"])  # 指定顺序
    .reset_index()
)
funnel_uv.columns = ["behavior_type", "uv"]
print(funnel_uv)

  behavior_type      uv
0            pv  983847
1           fav  366998
2          cart  706577
3           buy  626650


In [4]:
df.to_csv('../UserBehavior_cleaning.csv',index=False)